# Exploring Zotero MCP — Local Library Interaction

This notebook walks through how the Zotero MCP tools interact with a local Zotero library,
step by step. You'll trace the same path the code takes when an AI assistant calls a tool.

**Prerequisites:**
- Zotero desktop app is **running** (needed for the local API on port 23119)
- This project is installed in dev mode (`uv pip install -e ".[dev]" ipykernel`)

## Step 0: Set Up Environment

The MCP tools read `ZOTERO_LOCAL` to decide whether to use the local API or the web API.
We set it here so all subsequent cells use local mode.

In [1]:
import os

# Tell pyzotero to use the local Zotero API (localhost:23119)
os.environ["ZOTERO_LOCAL"] = "true"
os.environ["ZOTERO_LIBRARY_ID"] = "0"  # default for local
os.environ["ZOTERO_LIBRARY_TYPE"] = "user"

## Step 1: Create a Zotero Client

This is what every tool does first — call `get_zotero_client()` from `client.py`.
With `ZOTERO_LOCAL=true`, it creates a pyzotero client pointing at `localhost:23119`.

In [2]:
from zotero_mcp.client import get_zotero_client

zot = get_zotero_client()

# Inspect what we got
print(f"Library ID:   {zot.library_id}")
print(f"Library type: {zot.library_type}")
print(f"Local mode:   {zot.local}")

Library ID:   0
Library type: users
Local mode:   True


In [3]:
zot.collections()

[{'key': '53MQKF6F',
  'version': 400,
  'library': {'type': 'user',
   'id': 8745279,
   'name': 'My Library',
   'links': {'self': {'href': 'http://localhost:23119/api/users/8745279',
     'type': 'application/json'},
    'alternate': {'href': 'https://www.zotero.org/users/8745279',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'http://localhost:23119/api/users/8745279/collections/53MQKF6F',
    'type': 'application/json'},
   'alternate': {'href': 'https://www.zotero.org/users/8745279/collections/53MQKF6F',
    'type': 'text/html'}},
  'meta': {'numCollections': 0, 'numItems': 9},
  'data': {'key': '53MQKF6F',
   'version': 400,
   'name': 'Books',
   'parentCollection': False,
   'relations': {}}},
 {'key': 'VSA3YVNF',
  'version': 14067,
  'library': {'type': 'user',
   'id': 8745279,
   'name': 'My Library',
   'links': {'self': {'href': 'http://localhost:23119/api/users/8745279',
     'type': 'application/json'},
    'alternate': {'href': 'https://www.zotero.org/user

In [4]:
zot.collection_items_top('9KKKICYD')

[{'key': '9QCLKRJ8',
  'version': 15148,
  'library': {'type': 'user',
   'id': 8745279,
   'name': 'My Library',
   'links': {'self': {'href': 'http://localhost:23119/api/users/8745279',
     'type': 'application/json'},
    'alternate': {'href': 'https://www.zotero.org/users/8745279',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'http://localhost:23119/api/users/8745279/items/9QCLKRJ8',
    'type': 'application/json'},
   'alternate': {'href': 'https://www.zotero.org/users/8745279/items/9QCLKRJ8',
    'type': 'text/html'},
   'attachment': {'href': 'http://localhost:23119/api/users/8745279/items/7RHLTVTY',
    'type': 'application/json',
    'attachmentType': 'application/pdf',
    'attachmentSize': 20262988}},
  'meta': {'creatorSummary': 'Dominguez Mantes et al.',
   'parsedDate': '2025-07',
   'numChildren': 1},
  'data': {'key': '9QCLKRJ8',
   'version': 15148,
   'itemType': 'journalArticle',
   'title': 'Spotiflow: accurate and efficient spot detection for fluoresc

## Step 2: Search Items (what `zotero_search_items` does)

The tool calls `zot.add_parameters(...)` then `zot.items()`.
Under the hood, pyzotero sends:
```
GET http://localhost:23119/api/users/0/items?q=...&qmode=...&limit=...
```

Try changing the query to something in your library!

In [5]:
# --- Replicate what zotero_search_items does ---
query = "machine learning"  # <-- change this to match your library
limit = 3

zot.add_parameters(q=query, qmode="titleCreatorYear", itemType="-attachment", limit=limit)
results = zot.items()

print(f"Found {len(results)} results for '{query}'\n")

# Show the raw JSON structure of the first result
if results:
    import json
    print("--- Raw JSON structure (first result, data keys) ---")
    print(json.dumps(list(results[0]["data"].keys()), indent=2))

Found 3 results for 'machine learning'

--- Raw JSON structure (first result, data keys) ---
[
  "key",
  "version",
  "itemType",
  "title",
  "creators",
  "tags",
  "collections",
  "relations",
  "dateAdded",
  "dateModified"
]


## Step 3: Inspect a Result

Each result is a nested dict: `{"key": "...", "data": {...}, "meta": {...}}`.
The tool extracts fields from the `data` sub-dict and formats them as markdown.

In [6]:
from zotero_mcp.utils import format_creators

if results:
    item = results[0]
    data = item["data"]

    print(f"Title:     {data.get('title', 'Untitled')}")
    print(f"Item Key:  {item.get('key')}")
    print(f"Type:      {data.get('itemType')}")
    print(f"Date:      {data.get('date', 'No date')}")
    print(f"Authors:   {format_creators(data.get('creators', []))}")
    print(f"Tags:      {[t['tag'] for t in data.get('tags', [])]}")

    if abstract := data.get("abstractNote"):
        print(f"Abstract:  {abstract[:150]}...")
else:
    print("No results — try a different query in the cell above.")

Title:     Linear Algebra and Optimization for Machine Learning
Item Key:  J22G72QX
Type:      book
Date:      No date
Authors:   No authors listed
Tags:      []


## Step 4: Format as Markdown (the tool's output)

The tool uses `format_item_metadata()` from `client.py` for detailed views.
Let's see what the AI assistant actually receives back.

In [7]:
from zotero_mcp.client import format_item_metadata
from IPython.display import Markdown

if results:
    md_output = format_item_metadata(results[0])
    display(Markdown(md_output))
else:
    print("No results to format.")

# Linear Algebra and Optimization for Machine Learning

**Type:** book

**Item Key:** J22G72QX

**Collections:** 1 collections

**Notes/Attachments:** 1

## Step 5: Other Common Operations

Here are the building blocks the other tools use.
Uncomment and run any section you want to try.

In [8]:
# --- Get collections ---
collections = zot.collections()
for c in collections[:5]:
    print(f"  {c['data']['name']} (key={c['key']}, items={c['meta'].get('numItems', '?')})")

  Books (key=53MQKF6F, items=9)
  Chemical Biology (key=VSA3YVNF, items=5)
  Proteomics (key=5EVT7DL7, items=1)
  RNA Localized Translation (key=VZ7EU6C4, items=34)
  Cotranslational Assembly (key=VZ3JVJTK, items=10)


In [9]:
# --- Get tags ---
tags = zot.tags()
print(f"Total tags: {len(tags)}")
print(f"First 10:  {tags[:10]}")

Total tags: 100
First 10:  ['/unread', '2-HG', '2-hydroxyglutarate', "3' Untranslated Regions", "3' UTR", '3-D reconstruction', '3D cell culture', '3D genome organization', '3D models', "5' Untranslated Regions"]


In [10]:
# --- Get recent items ---
zot.add_parameters(sort="dateAdded", direction="desc", limit=3, itemType="-attachment")
recent = zot.items()
for r in recent:
    print(f"  [{r['data'].get('dateAdded', '?')[:10]}] {r['data'].get('title', 'Untitled')}")

  [2026-02-10] Impact and correction of segmentation errors in spatial transcriptomics
  [2026-02-01] Practical sensorless aberration estimation for 3D microscopy with deep learning
  [2026-02-01] Neuronal vulnerability and multilineage diversity in multiple sclerosis


In [11]:
# --- Get children (attachments/notes) for an item ---
if results:
    item_key = results[0]["key"]
    children = zot.children(item_key)
    print(f"Children of {item_key}:")
    for child in children:
        cd = child["data"]
        print(f"  {cd['itemType']}: {cd.get('title', cd.get('filename', 'untitled'))}")

Children of J22G72QX:
  attachment: PDF


---

## Step 5b: Accessing Group / Shared Libraries

The local API supports group libraries using the same URL pattern as the web API.
The key difference: `library_type="group"` and `library_id` = the **group ID** (not 0).

```
User library:  GET http://localhost:23119/api/users/0/items
Group library: GET http://localhost:23119/api/groups/1234567/items
```

In [12]:
# Discover all synced group libraries via the local API
from pyzotero import zotero

user_zot = zotero.Zotero(library_id="0", library_type="user", local=True)
groups = user_zot.groups()

print(f"Found {len(groups)} group libraries:\n")
for g in groups:
    gdata = g.get("data", g)  # structure may vary
    group_id = g.get("id", gdata.get("id", "?"))
    name = gdata.get("name", "?")
    print(f"  Group ID: {group_id}")
    print(f"  Name:     {name}")
    print()

Found 3 group libraries:

  Group ID: 6069773
  Name:     CNS-omics-review

  Group ID: 6098128
  Name:     SCZ

  Group ID: 6384606
  Name:     whole-cell-model



In [13]:
# Alternative: find groups directly from the SQLite database (Zotero doesn't need to be running)
# immutable=1 bypasses Zotero's database lock so we can read while Zotero is open
import sqlite3
from pathlib import Path

db_path = Path.home() / "Zotero" / "zotero.sqlite"
conn = sqlite3.connect(f"file:{db_path}?immutable=1", uri=True)
conn.row_factory = sqlite3.Row

rows = conn.execute("SELECT groupID, libraryID, name FROM groups").fetchall()
print(f"Groups in SQLite ({len(rows)}):\n")
for row in rows:
    print(f"  groupID={row['groupID']}  libraryID={row['libraryID']}  name={row['name']}")

conn.close()

Groups in SQLite (3):

  groupID=6069773  libraryID=19  name=CNS-omics-review
  groupID=6098128  libraryID=20  name=SCZ
  groupID=6384606  libraryID=24  name=whole-cell-model


In [14]:
# Now search within a group library
# Replace with a group ID from the output above
GROUP_ID = 6384606  # <-- paste your group ID here

group_zot = zotero.Zotero(library_id=str(GROUP_ID), library_type="group", local=True)

# List a few items from the group
group_zot.add_parameters(limit=5, itemType="-attachment", sort="dateModified", direction="desc")
group_items = group_zot.items()

print(f"Recent items in group {GROUP_ID}:\n")
for item in group_items:
    d = item["data"]
    print(f"  [{d.get('itemType')}] {d.get('title', 'Untitled')}")
    print(f"    key={item['key']}")

Recent items in group 6384606:

  [journalArticle] A Whole-Cell Computational Model Predicts Phenotype from Genotype
    key=SPZFCBVH


---

## Step 6: Direct SQLite Access (the other local path)

The semantic search system bypasses the HTTP API entirely and reads
`~/Zotero/zotero.sqlite` directly. This is faster for bulk operations.

**Note:** Zotero does NOT need to be running for this path.

In [15]:
from zotero_mcp.local_db import LocalZoteroReader

reader = LocalZoteroReader()
print(f"Database:    {reader.db_path}")
print(f"Item count:  {reader.get_item_count()}")

Database:    /Users/jiahao/Zotero/zotero.sqlite
Item count:  4650


In [16]:
# Fetch a few items via direct SQL
items = reader.get_items_with_text(limit=3)

for item in items:
    print(f"  [{item.item_type}] {item.title}")
    print(f"    key={item.key}, creators={item.creators}")
    print(f"    has abstract: {bool(item.abstract)}, has notes: {bool(item.notes)}")
    print()

  [journalArticle] Hunter-gatherers took refuge in European ‘water world’ for millennia
    key=L377QHSY, creators=Callaway, Ewen
    has abstract: True, has notes: False

  [journalArticle] US repeals key ‘endangerment finding’ that climate change is a public threat
    key=8DUF8GPT, creators=Witze, Alexandra
    has abstract: True, has notes: False

  [journalArticle] Daily briefing: Hunter-gatherers in Europe’s ‘water world’ resisted the switch to farming for millennia
    key=PF22UPWM, creators=Smith, Jacob
    has abstract: True, has notes: False



In [17]:
# Compare: same search, two different paths
search_query = "machine learning"  # <-- use the same query as Step 2

# Path A: pyzotero local API (HTTP)
zot2 = get_zotero_client()
zot2.add_parameters(q=search_query, qmode="everything", limit=5, itemType="-attachment")
api_results = zot2.items()

# Path B: direct SQLite
sql_results = reader.search_items_by_text(search_query, limit=5)

print(f"Local API found: {len(api_results)} items")
print(f"Direct SQL found: {len(sql_results)} items")
print()
print("API results:")
for r in api_results:
    print(f"  {r['data'].get('title', 'Untitled')}")
print()
print("SQL results:")
for r in sql_results:
    print(f"  {r.title}")

Local API found: 5 items
Direct SQL found: 5 items

API results:
  Spotiflow: accurate and efficient spot detection for fluorescence microscopy with deep stereographic flow regression
  Untitled
  CelloType: a unified model for segmentation and classification of tissue images
  Revealing a coherent cell-state landscape across single-cell datasets with CONCORD
  Artificial intelligence agents for biology

SQL results:
  Machine learning for predicting CKD stages in patients with autosomal dominant polycystic kidney disease: a nationwide cohort study in Japan
  Integrative transcriptomic and machine learning framework reveals candidate genes and potential mechanisms of aflatoxin B1 exposure in breast cancer
  Harvesting insights: interpretable machine learning to understand environmental drivers of U.S. maize and soybean yield
  Assessing extracellular vesicle proteins as predictive biomarkers for developing type 1 diabetes
  Cellular Aging Signatures in the Plasma Proteome Record Human 

In [18]:
# Clean up
reader.close()

## What You've Learned

1. **`get_zotero_client()`** reads env vars and creates a pyzotero client (local or remote)
2. **Local API path**: pyzotero sends HTTP requests to `localhost:23119` — Zotero must be running
3. **Direct SQLite path**: `LocalZoteroReader` opens `zotero.sqlite` read-only — used for bulk semantic search indexing
4. **Tool output**: raw JSON → extract `data` fields → format as markdown string → return to AI assistant

### Next steps to try:
- Modify the search query to explore your own library
- Look at `server.py` tool functions like `zotero_get_annotations` or `zotero_get_item_fulltext`
- Try the `generate_bibtex()` function from `client.py` on a result